# SO(3) benchmarking and profiling

Runs the generic benchmark helper with an explicit SO(3) backend, then profiles the same benchmark call with the standard-library `cProfile` profiler.


In [6]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
try:
    candidates.extend(path for path in start.iterdir() if path.is_dir())
except OSError:
    pass

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "pyproject.toml").exists()
        and (path / "src" / "equivariant_polynomials").is_dir()
        and (path / "benchmarks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find the project root. Start Jupyter from the project directory "
        "or from its notebooks directory."
    )

for import_path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

print(f"Project root: {PROJECT_ROOT}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: C:\Users\micha\OneDrive\Desktop\Oden\Notes\Equivariant Polynomial Tensor Functions\Python\equivariant_polynomials


In [7]:
import cProfile
import io
import pstats

from pprint import pp

from benchmarks import benchmark
from equivariant_polynomials.groups.so3 import SO3RepresentationTheory, hilbert_series_so3, hilbert_series_so3_multigraded


In [8]:
random_seed = 12345
input_irreps = (2, 3)
input_multiplicities = (2, 1)
output_irreps = (4,)
max_degree = 9
trivial_irrep = 0
modulus = 2521
profile_sort = "cumtime"
profile_rows = 25

benchmark_config = {
    "hilbert_series": hilbert_series_so3,
    "hilbert_series_multigraded": hilbert_series_so3_multigraded,
    "input_irreps": input_irreps,
    "input_multiplicities": input_multiplicities,
    "max_degree": max_degree,
    "trivial_irrep": trivial_irrep,
    "random_seed": random_seed,
    "modulus": modulus,
}


## Benchmark


In [9]:
theory = SO3RepresentationTheory()
summaries = []

for output_irrep in output_irreps:
    print(f"\nRun: output_irrep={output_irrep}, max_degree={max_degree}")
    summaries.append(
        benchmark(
            theory=theory,
            output_irrep=output_irrep,
            **benchmark_config,
        )
    )

summaries = tuple(summaries)
pp(summaries)



Run: output_irrep=4, max_degree=9
algebra generators: 2.500 s
module generators: 6.406 s
total: 8.924 s
({'scenario': {'input_irreps': (2, 3),
               'input_multiplicities': (2, 1),
               'output_irrep': 4,
               'trivial_irrep': 0,
               'max_degree': 9,
               'random_seed': 12345,
               'modulus': 2521},
  'algebra': {'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739),
              'generator_counts': (0, 0, 4, 7, 14, 26, 52, 68, 60, 39),
              'matches_known_generators': True,
              'generator_seconds': 2.4999477000092156},
  'module': {'hilbert_dimensions': (0,
                                    0,
                                    6,
                                    21,
                                    87,
                                    273,
                                    807,
                                    2109,
                                    5205,
                    

## cProfile


In [10]:
profiled_output_irrep = output_irreps[0]
profiler = cProfile.Profile()
profile_summary = profiler.runcall(
    benchmark,
    theory=SO3RepresentationTheory(),
    output_irrep=profiled_output_irrep,
    **benchmark_config,
)

stats_stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stats_stream)
stats.strip_dirs().sort_stats(profile_sort).print_stats(profile_rows)
print(stats_stream.getvalue())

profile_summary


algebra generators: 2.986 s
module generators: 6.856 s
total: 9.896 s
         3475790 function calls (3463486 primitive calls) in 9.866 seconds

   Ordered by: cumulative time
   List reduced from 391 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        5    0.011    0.002    9.827    1.965 selectors.py:310(select)
        1    0.001    0.001    6.880    6.880 benchmark.py:1(benchmark)
      248    0.012    0.000    5.547    0.022 generators.py:1(_stream_content_generators)
      917    0.004    0.000    4.678    0.005 generators.py:1(_syndrome_batches)
      707    0.017    0.000    4.327    0.006 generators.py:73(compute_syndromes)
     5461    0.132    0.000    4.308    0.001 evaluators.py:483(evaluate_basis)
      421    0.122    0.000    4.056    0.010 generators.py:1(_previous_content_product_basis)
     4531    0.093    0.000    3.678    0.001 evaluators.py:377(evaluate_young_symmetrized_tensor_tree)
      794    2.586  

{'scenario': {'input_irreps': (2, 3),
  'input_multiplicities': (2, 1),
  'output_irrep': 4,
  'trivial_irrep': 0,
  'max_degree': 9,
  'random_seed': 12345,
  'modulus': 2521},
 'algebra': {'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739),
  'generator_counts': (0, 0, 4, 7, 14, 26, 52, 68, 60, 39),
  'matches_known_generators': True,
  'generator_seconds': 2.9864480999531224},
 'module': {'hilbert_dimensions': (0,
   0,
   6,
   21,
   87,
   273,
   807,
   2109,
   5205,
   11919),
  'generator_counts': (0, 0, 6, 21, 63, 147, 264, 284, 133, 35),
  'matches_known_generators': True,
  'generator_seconds': 6.856331299990416},
 'total_seconds': 9.896217399975285}